In [1]:
import pandas as pd
from tqdm import tqdm

In [3]:
VCF_FILE = "../data/raw/1000genomes/ALL.chr3.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf"

CLINVAR_FILE = "../data/processed/clinvar_filtered.csv"

SAVE_FILE = "../data/processed/patient_gene_mutation_counts_chr3.csv"

Load IBDC Variant Positions

In [6]:
clinvar = pd.read_csv(CLINVAR_FILE)

clinvar_positions = set(
    zip(clinvar["Chromosome"].astype(str),
        clinvar["Start"].astype(int))
)

print("clinvar variants:", len(clinvar_positions))

clinvar variants: 1435118


Read VCF Header (Extract Patient Names)

In [ ]:
with open(VCF_FILE) as f:
    for line in f:
        if line.startswith("#CHROM"):
            header = line.strip().split("\t")
            break

samples = header[9:]

print("Total patients:", len(samples))

Stream Through the VCF File

In [32]:
gene_counts = {}

with open(VCF_FILE) as f:

    for line in tqdm(f):

        if line.startswith("#"):
            continue

        fields = line.strip().split("\t")

        chrom = fields[0]
        pos = int(fields[1])

        if (chrom, pos) not in clinvar_positions:
            continue

        genotypes = fields[9:]

        for i,gt in enumerate(genotypes):

            gt = gt.split(":")[0]

            if gt not in ["0|0","0/0","."]:

                patient = samples[i]

                key = (patient, chrom, pos)

                gene_counts[key] = gene_counts.get(key, 0) + 1

583213it [01:02, 9385.69it/s] 


Convert to DataFrame

In [33]:
data = []

for (patient,chrom,pos),count in gene_counts.items():

    data.append({
        "patient": patient,
        "chrom": chrom,
        "pos": pos,
        "mutation_count": count
    })

variant_df = pd.DataFrame(data)

variant_df.head()

,patient,chrom,pos,mutation_count
0,HG00240,3,382235,1
1,HG00343,3,382235,1
2,HG01082,3,382235,1
3,HG01605,3,382235,1
4,HG03108,3,382235,1


Save Processed Dataset

In [34]:
variant_df.to_csv(SAVE_FILE, index=False)

print("Saved:", SAVE_FILE)

Saved: ../data/processed/patient_gene_mutation_counts_chr3.csv


Merge the Chromosome Results

In [35]:
import pandas as pd

chr1 = pd.read_csv("../data/processed/patient_gene_mutation_counts_chr1.csv")
chr3 = pd.read_csv("../data/processed/patient_gene_mutation_counts_chr3.csv")

combined = pd.concat([chr1, chr3], ignore_index=True)

combined.to_csv("../data/processed/patient_gene_mutation_counts.csv", index=False)

print("Combined dataset size:", combined.shape)

Combined dataset size: (102676, 4)
